# Health IT and health-tech AI lab for medicos

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zistgah/hecon/blob/main/notebooks/04_health_it_ai_lab.ipynb)

Hands-on steps behind sections 4 and 7 of the paper, on a synthetic case (no real person): de-identification before anything is derived, re-identification risk, dialysis adequacy and kidney function, structured reasoning records, a Socratic AI tutor with a keyless path, a grounding check on AI output and a FHIR R4 export.

**How to use it.** Press ▶ on each step in order, or choose Runtime, then Run all. Each step's code is hidden behind its title; double-click the title to see it. Educational use only: nothing here is a medical device or a finding.

© 1993–2026 Abhishek Choudhary. All rights reserved. AyeAI. Code GPL-3.0-or-later; text CC BY-SA 4.0. AyeAI® and AyeAM® are registered trade marks of Abhishek Choudhary; AyeCNSe™ (Indian TM application No. 7415871, pending). Cite with [CITATION.cff](https://github.com/zistgah/hecon/blob/main/CITATION.cff); the DOI is recorded in the repository README once minted.

**Where this notebook lives.** It is hosted in zistgah/hecon until it moves to its home, zistgah/marham, the estate's health-informatics layer. The move and the link back are listed in [COMPONENTS.md](https://github.com/zistgah/hecon/blob/main/COMPONENTS.md).

## Global privacy, by default

The steps follow the strictest of the major frameworks, so that they hold wherever a case comes from: the EU GDPR (Regulation (EU) 2016/679), the US HIPAA Privacy Rule (45 CFR 164.514(b), Safe Harbor and Expert Determination), the UK Data Protection Act 2018, India's Digital Personal Data Protection Act 2023 and ICMR 2017 guidelines, Brazil's LGPD (Law 13.709/2018), South Africa's POPIA (2013) and Kenya's Data Protection Act 2019, with the Declaration of Helsinki (2024), the CIOMS guidelines (2016) and WHO guidance on AI for health (2021). Where two of them differ, the stricter requirement is used. This notebook teaches; it is not legal advice.

In [ ]:
#@title Set up: load the hecon package from the repository { display-mode: "form" }
import subprocess, sys
REPO = "zistgah/hecon"  #@param {type:"string"}
try:
    import hecon
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/%s@main" % REPO], check=True)
    import hecon
print(hecon.NOTICE)

## Step 1: the synthetic case

A chronic kidney disease patient on haemodialysis, the paper's own example (section 4.1). Every name, number and address in it is invented.

In [ ]:
#@title Show the case { display-mode: "form" }
from hecon import health
print(health.CASE_TEXT)

## Step 2: screen before deriving

Identifiers are removed while the text is intact, before anything is extracted from it. Removing names is not enough (section 7.1): record numbers, dates, addresses, ages over 89 and contact details identify people too. National numbers are checked with their own checksums (NHS modulus 11, Verhoeff for 12-digit identity numbers), and a number that fails the checksum is still removed.

In [ ]:
#@title De-identify and measure { display-mode: "form" }
clean, report = health.deidentify(health.CASE_TEXT, names=health.CASE_NAMES)
print(clean); print(); print("Found:", report["counts"])
ev = health.evaluate(clean, health.GOLD_IDENTIFIERS, health.CLINICAL_VALUES)
print("Identifiers removed: %d of %d; clinical values kept: %d of %d" % (ev["removed"], ev["identifiers"], ev["retained"], ev["clinical_values"]))
print("Missed:", ev["missed"] or "none", "| Lost:", ev["lost"] or "none")

## Step 3: shifting dates keeps the intervals

For research records, dates can be moved by one secret offset per patient, so intervals between events survive.

In [ ]:
#@title Shift dates by an offset { display-mode: "form" }
offset_days = 137  #@param {type:"integer"}
print(health.deidentify(health.CASE_TEXT, names=health.CASE_NAMES, date_shift_days=offset_days)[0][:420])

## Step 4: re-identification risk in a table

*k-anonymity* is the size of the smallest group sharing the same quasi-identifiers; k = 1 means someone is unique. *l-diversity* asks whether each group still holds more than one diagnosis. Generalising age and region raises k.

In [ ]:
#@title k-anonymity and l-diversity { display-mode: "form" }
rows = health.synthetic_cohort(); qis = ["age", "sex", "region"]
print("raw: k =", health.k_anonymity(rows, qis))
for band, merge in [(10, False), (30, False), (60, True)]:
    g = health.generalise(rows, age_band=band, region_map={r: "all regions" for r in ("North", "South", "East", "West")} if merge else None)
    print("age band %d, regions merged %s: k = %d, l = %d" % (band, merge, health.k_anonymity(g, qis), health.l_diversity(g, qis, "diagnosis")))

## Step 5: kidney function and dialysis adequacy

eGFR by the CKD-EPI 2021 creatinine equation without a race term (Inker et al., 2021), the urea reduction ratio, and single-pool Kt/V by the second-generation Daugirdas formula. Education only: check against your local laboratory and guidelines.

In [ ]:
#@title Calculate { display-mode: "form" }
c = health.CASE
egfr = health.egfr_ckd_epi_2021(health.creatinine_umol_to_mgdl(c["creatinine_umol_l"]), c["age"], female=False)
print("eGFR: %.1f mL/min/1.73 m2" % egfr)
print("URR: %.1f %%" % health.urr(c["urea_pre"], c["urea_post"]))
print("Kt/V: %.2f" % health.ktv_daugirdas2(c["urea_pre"], c["urea_post"], c["session_hours"], c["uf_litres"], c["post_weight_kg"]))

## Step 6: a structured reasoning record

The record captures what a learner states, not private thought (section 4.2). It is checked against a schema and matched to the case's key features. The match is a crude aid for a teacher, never a grade on its own.

In [ ]:
#@title Validate and score a record { display-mode: "form" }
import json
record = {"learner": "L-017", "case_id": health.CASE["id"],
          "problem_representation": "Man on maintenance haemodialysis with hyperkalaemia and palpitations after a missed session",
          "differentials": [{"diagnosis": "hyperkalaemia from a missed session", "for": ["potassium 6.4", "missed session"], "against": []}],
          "information_sought": ["ECG", "medicine history", "diet"],
          "decisions": [{"decision": "urgent dialysis", "rationale": "potassium 6.4 mmol/L with palpitations"}],
          "uncertainty": "ECG not yet seen; medicine list incomplete"}
print("Schema problems:", health.validate_record(record) or "none")
print(json.dumps(health.score_record(record), indent=1))

## Step 7: a Socratic question ladder

Questions mapped to the five domains of the UK clinical reasoning consensus (Cooper et al., 2021) and to cognitive levels.

In [ ]:
#@title Show the ladder { display-mode: "form" }
for dom, level, q in health.socratic_ladder(): print("%-45s %-9s %s" % (dom, level, q))

## Step 8: an AI tutor, with no lock-in

Choose a provider. *rule-based* needs no model, no key and no network. *local* runs an open-weight model in this notebook (Qwen2.5 0.5B on any runtime; the same family the Hindawi LLM notebooks use). *endpoint* uses any OpenAI-compatible service with your own key, typed when asked and never stored. Whatever answers, the grounding check lists every number the tutor wrote that is not in the case, for a teacher to check.

In [ ]:
#@title Ask the tutor { display-mode: "form" }
provider = "rule-based"  #@param ["rule-based", "local", "endpoint"]
local_model = "Qwen/Qwen2.5-0.5B-Instruct"  #@param ["Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-7B-Instruct"]
endpoint_url = "http://localhost:11434/v1"  #@param {type:"string"}
endpoint_model = "llama3.1"  #@param {type:"string"}
learner_says = "I think this is fluid overload, so I would give diuretics."  #@param {type:"string"}
if provider == "local":
    p = health.LocalProvider(local_model)
elif provider == "endpoint":
    from getpass import getpass
    p = health.EndpointProvider(endpoint_url, endpoint_model, getpass("API key (leave empty if none): "))
else:
    p = health.RuleProvider()
answer = p.complete(health.tutor_messages(health.deidentify(health.CASE_TEXT, names=health.CASE_NAMES)[0], learner_says))
print("[%s]" % p.name, answer)
print("Numbers to check against the case:", health.grounding(answer, health.CASE_TEXT) or "none")

## Step 9: a FHIR R4 export

The de-identified case as a FHIR Bundle. Codes are left out on purpose: a code comes from a terminology service, never from memory. Set *lookup* to search a public FHIR terminology server for candidates; that step needs the internet and was not run when this notebook was built.

In [ ]:
#@title Build, validate and optionally look up codes { display-mode: "form" }
import json, subprocess, sys
bundle = health.fhir_bundle()
print(json.dumps(bundle, indent=1)[:900], "...")
try:
    try:
        import fhir.resources  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fhir.resources"], check=True)
    from fhir.resources.R4B.bundle import Bundle
    (Bundle.model_validate if hasattr(Bundle, "model_validate") else Bundle.parse_obj)(bundle)
    print("Validated against the FHIR R4B models in fhir.resources.")
except Exception as e:
    print("Not validated here:", e.__class__.__name__, str(e)[:160])
lookup = False  #@param {type:"boolean"}
term = "potassium serum"  #@param {type:"string"}
if lookup:
    import urllib.parse, urllib.request
    url = "https://tx.fhir.org/r4/ValueSet/$expand?" + urllib.parse.urlencode({"url": "http://loinc.org/vs", "filter": term, "count": 5})
    vs = json.load(urllib.request.urlopen(urllib.request.Request(url, headers={"Accept": "application/fhir+json"}), timeout=60))
    for c in vs.get("expansion", {}).get("contains", []): print(c.get("system"), c.get("code"), c.get("display"))

## Step 10: keep what you made

The de-identified case, the record and the bundle are written to a folder, and offered for download in Colab.

In [ ]:
#@title Save the artefacts { display-mode: "form" }
import os
os.makedirs("hecon_outputs", exist_ok=True)
open("hecon_outputs/case_deidentified.txt", "w").write(clean)
json.dump(record, open("hecon_outputs/reasoning_record.json", "w"), indent=1)
json.dump(bundle, open("hecon_outputs/fhir_bundle.json", "w"), indent=1)
print(sorted(os.listdir("hecon_outputs")))
try:
    from google.colab import files
    for n in sorted(os.listdir("hecon_outputs")): files.download("hecon_outputs/" + n)
except ImportError:
    pass

## What to do next

To use real cases: obtain the permissions your setting requires, run Step 2 before anything else, check k on any table you release, and have a teacher review every AI-generated clinical statement (paper, section 4.4).